In [7]:
import sqlite3
import pandas as pd
import time
from nba_api.stats.endpoints import shotchartdetail

In [10]:
conn = sqlite3.connect('nba_centers_info.db')

def get_shot_loc(start,end):
    desired_seasons = [f"{year}-{str(year+1)[2:]}" for year in range(start, end + 1)]

    seasons = desired_seasons

    shot_loc = []

    for season in seasons: 
        print(f"Fetching player shot locations for {season}...")

        try:
            info = shotchartdetail.ShotChartDetail(
                team_id = 0,
                player_id = 0,
                season_nullable=season, 
                context_measure_simple='FGA', 
                season_type_all_star='Regular Season')
            df = info.get_data_frames()[0]

            desired_info = df[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_NAME', 'PERIOD', 
                               'EVENT_TYPE', 'SHOT_TYPE', 'SHOT_DISTANCE', 'LOC_X', 'LOC_Y']]

            shot_loc.append(desired_info)

            time.sleep(1)

        except Exception as e:
            print(f"Error {e}")

    if shot_loc:
        final_df = pd.concat(shot_loc, ignore_index=True)
        
        # Add to the 'ShotLocations' table
        final_df.to_sql('ShotLocs', conn, if_exists='append', index=False)
        print(final_df)


In [11]:
get_shot_loc(2013, 2024)

Fetching player shot locations for 2013-14...
Fetching player shot locations for 2014-15...
Fetching player shot locations for 2015-16...
Fetching player shot locations for 2016-17...
Fetching player shot locations for 2017-18...
Fetching player shot locations for 2018-19...
Fetching player shot locations for 2019-20...
Fetching player shot locations for 2020-21...
Fetching player shot locations for 2021-22...
Fetching player shot locations for 2022-23...
Fetching player shot locations for 2023-24...
Fetching player shot locations for 2024-25...
         PLAYER_ID       PLAYER_NAME              TEAM_NAME  PERIOD  \
0             2561        David West         Indiana Pacers       1   
1           202331       Paul George         Indiana Pacers       1   
2           201167     Arron Afflalo          Orlando Magic       1   
3             2561        David West         Indiana Pacers       1   
4           202362  Lance Stephenson         Indiana Pacers       1   
...            ...    

In [12]:
conn = sqlite3.connect('nba_centers_info.db')

shot_loc_df = pd.read_sql("SELECT * FROM ShotLocs", conn)
shot_loc_df

,PLAYER_ID,PLAYER_NAME,TEAM_NAME,PERIOD,EVENT_TYPE,SHOT_TYPE,SHOT_DISTANCE,LOC_X,LOC_Y
0,2561,David West,Indiana Pacers,1,Missed Shot,2PT Field Goal,5.0,-38.0,45.0
1,202331,Paul George,Indiana Pacers,1,Made Shot,2PT Field Goal,19.0,105.0,164.0
2,201167,Arron Afflalo,Orlando Magic,1,Missed Shot,3PT Field Goal,27.0,51.0,266.0
3,2561,David West,Indiana Pacers,1,Missed Shot,2PT Field Goal,2.0,28.0,-5.0
4,202362,Lance Stephenson,Indiana Pacers,1,Made Shot,3PT Field Goal,26.0,15.0,260.0
...,...,...,...,...,...,...,...,...,...
2510333,1630578,Alperen Sengun,Houston Rockets,4,Missed Shot,3PT Field Goal,25.0,-29.0,257.0
2510334,1627832,Fred VanVleet,Houston Rockets,4,Missed Shot,3PT Field Goal,22.0,-228.0,-17.0
2510335,1629652,Luguentz Dort,Oklahoma City Thunder,4,Made Shot,3PT Field Goal,24.0,186.0,167.0
2510336,1630224,Jalen Green,Houston Rockets,4,Made Shot,2PT Field Goal,2.0,12.0,16.0


In [13]:
conn = sqlite3.connect('nba_centers_info.db')

centers_df = pd.read_sql("SELECT * FROM Centers", conn)
centers_df

,PERSON_ID,DISPLAY_FIRST_LAST,POSITION,HEIGHT,TEAM_NAME
0,203500,Steven Adams,Center,6-11,Rockets
1,1628386,Jarrett Allen,Center,6-9,Cavaliers
2,1629028,Deandre Ayton,Center,7-0,Lakers
3,202687,Bismack Biyombo,Center,6-8,Spurs
4,203991,Clint Capela,Center,6-10,Rockets
...,...,...,...,...,...
696,1917,Wang Zhi-zhi,Center,7-1,
697,678,George Zidek,Center,7-0,
698,1627757,Stephen Zimmerman,Center,7-0,Magic
699,1627790,Ante Zizic,Center,6-10,Cavaliers


In [14]:
conn.close()